In [3]:
import subprocess
import cv2
import os
import numpy as np
from collections import deque

# Centroid Tracker Class
class CentroidTracker:
    def __init__(self, max_disappeared=50):
        self.next_object_id = 0
        self.objects = {}  # Stores object IDs and their centroids
        self.disappeared = {}  # Tracks how many frames an object has been missing
        self.max_disappeared = max_disappeared  # Maximum frames an object can be missing

    def register(self, centroid):
        # Assign a new ID to the object
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def deregister(self, object_id):
        # Remove an object ID from tracking
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update(self, rects):
        # If no detections, mark all objects as disappeared
        if len(rects) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects

        # Initialize an array of input centroids for the current frame
        input_centroids = np.zeros((len(rects), 2), dtype="int")

        # Loop over the bounding box rectangles and calculate centroids
        for (i, (start_x, start_y, end_x, end_y)) in enumerate(rects):
            c_x = int((start_x + end_x) / 2.0)
            c_y = int((start_y + end_y) / 2.0)
            input_centroids[i] = (c_x, c_y)

        # If no objects are being tracked, register all new objects
        if len(self.objects) == 0:
            for i in range(0, len(input_centroids)):
                self.register(input_centroids[i])
        else:
            # Grab the set of object IDs and corresponding centroids
            object_ids = list(self.objects.keys())
            object_centroids = list(self.objects.values())

            # Compute the Euclidean distance between each pair of object centroids and input centroids
            distances = np.linalg.norm(np.array(object_centroids)[:, np.newaxis] - input_centroids, axis=2)

            # Find the smallest distance for each row and sort the indexes
            rows = distances.min(axis=1).argsort()
            cols = distances.argmin(axis=1)[rows]

            # Track used rows and columns
            used_rows = set()
            used_cols = set()

            # Loop over the combination of (row, column) index tuples
            for (row, col) in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue

                # Assign the input centroid to the existing object
                object_id = object_ids[row]
                self.objects[object_id] = input_centroids[col]
                self.disappeared[object_id] = 0

                # Mark the row and column as used
                used_rows.add(row)
                used_cols.add(col)

            # Compute indexes NOT already examined
            unused_rows = set(range(0, distances.shape[0])) - used_rows
            unused_cols = set(range(0, distances.shape[1])) - used_cols

            # If there are more object centroids than input centroids, mark missing objects
            if distances.shape[0] >= distances.shape[1]:
                for row in unused_rows:
                    object_id = object_ids[row]
                    self.disappeared[object_id] += 1
                    if self.disappeared[object_id] > self.max_disappeared:
                        self.deregister(object_id)
            else:
                # Otherwise, register new objects
                for col in unused_cols:
                    self.register(input_centroids[col])

        return self.objects

# Function to parse Darknet output and extract bounding boxes
def parse_darknet_output(output):
    rects = []
    for line in output.splitlines():
        if "%" in line:  # Detection lines contain percentages
            parts = line.split()
            left = int(float(parts[2]))
            top = int(float(parts[3]))
            right = int(float(parts[4]))
            bottom = int(float(parts[5]))
            rects.append((left, top, right, bottom))
    return rects

# Paths
darknet_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/darknet"
cfg_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/cfg/yolov4-tiny-custom.cfg"
weights_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/TRAINING/yolov4-tiny/training/yolov4-tiny-custom_best.weights"
data_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/data/obj.data"
video_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/test.mp4"
output_dir = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/output"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Initialize video capture
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Error: Unable to open video file {video_path}")
    exit(1)

# Get the frame rate (frames per second) and total frames
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps  # Duration of the video in seconds

print(f"Video FPS: {fps}, Total Frames: {total_frames}, Duration: {duration} seconds")

# Initialize Centroid Tracker
ct = CentroidTracker(max_disappeared=50)

# Define frame interval (1 frame every 1 second)
frame_interval = fps  # Capture 1 frame every 1 second

frame_count = 0
saved_frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Exit the loop if no more frames are available

    # Save frame every 1 second
    if frame_count % frame_interval == 0:
        # Save the current frame
        frame_filename = os.path.join(output_dir, f"frame_{saved_frame_count:04d}.png")
        cv2.imwrite(frame_filename, frame)
        print(f"Saved frame to {frame_filename}")

        # Run Object Detection
        cmd = [
            darknet_path, 'detector', 'test', data_path, cfg_path, weights_path, frame_filename, '-thresh', '0.3'
        ]
        print(f"Running command: {' '.join(cmd)}")
        process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Print stdout and stderr to help debug
        stdout_output = process.stdout
        stderr_output = process.stderr

        print("STDOUT:\n", stdout_output)
        print("STDERR:\n", stderr_output)

        # Parse Darknet output to get bounding boxes
        rects = parse_darknet_output(stdout_output)

        # Update the centroid tracker with the detected bounding boxes
        objects = ct.update(rects)

        # Draw bounding boxes and object IDs on the frame
        for (object_id, centroid) in objects.items():
            text = f"ID {object_id}"
            cv2.putText(frame, text, (centroid[0] - 10, centroid[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            cv2.circle(frame, (centroid[0], centroid[1]), 4, (0, 255, 0), -1)

        # Save the frame with bounding boxes and IDs
        result_image_path = os.path.join(output_dir, f"frame_{saved_frame_count:04d}_predictions.jpg")
        cv2.imwrite(result_image_path, frame)
        print(f"Saved result image to {result_image_path}")

        saved_frame_count += 1

    frame_count += 1

# Release video capture and print completion message
cap.release()
print(f"Processing complete. Total frames processed: {frame_count}, Total frames saved: {saved_frame_count}")

Video FPS: 29, Total Frames: 1045, Duration: 36.03448275862069 seconds
Saved frame to /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/output/frame_0000.png
Running command: /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/darknet detector test /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/data/obj.data /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/cfg/yolov4-tiny-custom.cfg /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/TRAINING/yolov4-tiny/training/yolov4-tiny-custom_best.weights /mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/output/frame_0000.png -thresh 0.3
STDOUT:
  GPU isn't used 
mini_batch = 1, batch = 1, time_steps = 1, train = 0 
nms_kind: greedynms (1), beta = 

IndexError: list index out of range